# T02 - Cleaning (phần Python)
**SQL (file `01_cleaning.sql`) đã làm:** bỏ trùng, đổi giá, tính tuổi xe, lọc spam, tạo cờ, đổi mã sang chữ, chuẩn hóa tỉnh, bỏ ODO = 0, bỏ trùng xe.
**Python làm:** quyết định cắt hay không cắt ngoại lai (có biểu đồ giải thích), lọc IQR, kiểm tra, xuất `cars_cleaned.csv`.

Cần cài một lần: `pip install pandas numpy sqlalchemy pyodbc matplotlib ipykernel`

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import create_engine

SERVER = "localhost\\TEST"
engine = create_engine(
    f"mssql+pyodbc://@{SERVER}/UsedCars"
    "?driver=ODBC+Driver+18+for+SQL+Server"
    "&trusted_connection=yes&TrustServerCertificate=yes"
)
df = pd.read_sql("SELECT * FROM cars_clean", engine)
rows_from_sql = len(df)
print("Số dòng đọc từ SQL:", rows_from_sql)
print("ODO = 0 hoặc thiếu (phải = 0):", ((df["odo_km"] == 0) | df["odo_km"].isna()).sum())

## 1. Ngoại lai: xem dữ liệu trước khi quyết định
Quy tắc IQR: IQR = Q3 - Q1, giữ giá trị trong khoảng **Q1 - 1,5×IQR đến Q3 + 1,5×IQR**. Ở đây chỉ tính ngưỡng, chưa xóa gì.

In [ ]:
def iqr_bounds(s, floor=None):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    if floor is not None:
        lo = max(lo, floor)
    return lo, hi

p_lo, p_hi = iqr_bounds(df["price_million"], floor=20)
o_lo, o_hi = iqr_bounds(df["odo_km"], floor=0)
df["price_out"] = (df["price_million"] < p_lo) | (df["price_million"] > p_hi)
df["odo_out"] = (df["odo_km"] < o_lo) | (df["odo_km"] > o_hi)

print(f"Giá: ngưỡng IQR {p_lo:.0f} - {p_hi:.0f} triệu -> {df['price_out'].sum()} xe nằm ngoài ({df['price_out'].mean():.1%})")
print(f"ODO: ngưỡng IQR {o_lo:.0f} - {o_hi:.0f} km -> {df['odo_out'].sum()} xe nằm ngoài ({df['odo_out'].mean():.1%})")

### 1a. Giá: xe bị IQR loại là xe gì?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.8))

ax[0].hist(df["price_million"], bins=80, color="#8fa8c8")
ax[0].axvline(p_hi, color="red", ls="--", label=f"Ngưỡng IQR = {p_hi:.0f} triệu")
ax[0].set_title("Phân phối giá"); ax[0].set_xlabel("Giá (triệu)"); ax[0].legend()

for flag, color, label in [(False, "#8fa8c8", "Trong ngưỡng IQR"), (True, "red", "Bị IQR loại")]:
    s = df[df["price_out"] == flag]
    ax[1].scatter(s["car_age"], s["price_million"], s=6, alpha=0.5, c=color, label=label)
ax[1].set_yscale("log"); ax[1].set_xlabel("Tuổi xe"); ax[1].set_ylabel("Giá (triệu, thang log)")
ax[1].set_title("Giá theo tuổi xe"); ax[1].legend(markerscale=3)

top = df[df["price_out"]]["carbrand_name"].value_counts().head(8)
ax[2].barh(top.index[::-1], top.values[::-1], color="#c0504d")
ax[2].set_title("Hãng của xe bị IQR loại theo giá")
plt.tight_layout(); plt.show()

### 1b. Giá có "ngáo" không? So giá với các xe cùng hãng, cùng dòng
`ratio` = giá / giá trung vị của cùng dòng xe. Xe giá cao nhưng `ratio` gần 1 nghĩa là dòng xe đó vốn đắt (xe xịn, giá hợp lý). Chỉ xét dòng xe có ít nhất 10 tin.

In [ ]:
g = df.groupby(["carbrand_name", "carmodel_name"])["price_million"]
df["model_n"] = g.transform("size")
df["model_median"] = g.transform("median")
df["ratio"] = df["price_million"] / df["model_median"]
ok = df["model_n"] >= 10

out_ok = df[df["price_out"] & ok]
legit = (out_ok["ratio"].between(0.5, 2)).mean() if len(out_ok) else float("nan")
suspect = ok & ((df["ratio"] > 3) | (df["ratio"] < 1 / 3))
print(f"Xe bị IQR loại theo giá và có đủ dữ liệu so sánh: {len(out_ok)}")
print(f"-> trong đó {legit:.0%} có giá chỉ lệch tối đa 2 lần so với trung vị cùng dòng xe (nhiều khả năng là xe đắt thật)")
print(f"Xe nghi ngáo giá (giá > 3 lần hoặc < 1/3 trung vị cùng dòng): {suspect.sum()} ({suspect.sum() / ok.sum():.1%})")

fig, ax = plt.subplots(figsize=(8, 4.5))
bins = np.logspace(-1, 1, 70)
ax.hist(df.loc[ok & ~df["price_out"], "ratio"], bins=bins, alpha=0.6, label="Trong ngưỡng IQR", color="#8fa8c8")
ax.hist(df.loc[ok & df["price_out"], "ratio"], bins=bins, alpha=0.8, label="Bị IQR loại", color="red")
ax.set_xscale("log"); ax.axvline(1, color="k", ls=":")
ax.set_xlabel("Giá / trung vị cùng dòng xe (thang log)"); ax.set_ylabel("Số xe")
ax.set_title("Xe giá cao bị IQR loại có thật sự lệch giá không?"); ax.legend()
plt.show()

### 1c. ODO: xe bị IQR loại là xe gì?

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].hist(df["odo_km"], bins=80, color="#8fa8c8")
ax[0].axvline(o_hi, color="red", ls="--", label=f"Ngưỡng IQR = {o_hi:,.0f} km")
ax[0].set_title("Phân phối ODO"); ax[0].set_xlabel("ODO (km)"); ax[0].legend()
for flag, color, label in [(False, "#8fa8c8", "Trong ngưỡng IQR"), (True, "red", "Bị IQR loại")]:
    s = df[df["odo_out"] == flag]
    ax[1].scatter(s["car_age"], s["odo_km"], s=6, alpha=0.5, c=color, label=label)
ax[1].set_xlabel("Tuổi xe"); ax[1].set_ylabel("ODO (km)"); ax[1].set_title("ODO theo tuổi xe"); ax[1].legend(markerscale=3)
plt.tight_layout(); plt.show()

## 2. Quyết định
Điền kết luận sau khi xem biểu đồ ở trên. Gợi ý cách lập luận:
- **Giá:** nếu phần lớn xe bị loại có `ratio` gần 1 (xe sang, giá đúng) thì cắt bằng IQR sẽ làm mô hình mất nhóm xe đắt -> **không cắt**. Xe nghi ngáo giá chỉ chiếm tỷ lệ nhỏ, ghi nhận để xử lý sau.
- **ODO:** nếu điểm bị loại nằm ở ODO rất cao so với tuổi xe thì có thể là nhập sai -> **cắt**.

In [ ]:
CUT_PRICE = False   # True nếu muốn cắt giá bằng IQR
CUT_ODO = True      # True: cắt ODO bằng IQR

before = len(df)
if CUT_PRICE:
    df = df[~df["price_out"]]
if CUT_ODO:
    df = df[~df["odo_out"]]
print(f"Số dòng trước: {before} | sau: {len(df)} | đã loại: {before - len(df)}")

df = df.drop(columns=["price_out", "odo_out", "model_n", "model_median", "ratio"])
print(df[["price_million", "car_age", "odo_km"]].describe())

## 3. Xuất file cho nhóm

In [ ]:
df.to_csv("cars_cleaned.csv", index=False, encoding="utf-8-sig")
print("Đã lưu cars_cleaned.csv:", df.shape)
print(f"Tổng: {15000} dòng thô -> {rows_from_sql} sau SQL -> {len(df)} file cuối")